# Part 3 — Patient layers and cohort selection

We reload the PKN and lift the omics onto it: matched tumours become the
`patient` aspect, one Kivelä elementary layer each, and RNA-contrast, TF-activity
and kinase-activity land as vertex-layer attributes on the *same* graph. Then we
define the mutation-burden cohorts that `04` solves over.

In [ ]:
import numpy as np
import pandas as pd

from uc1_common import *  # noqa: F403

rna_contrast = pd.read_parquet(RNA_CONTRAST)
tf_es = pd.read_parquet(TF_ES)
kin_es = pd.read_parquet(KIN_ES)
G = load()

## Build patient layers

Vertex-layer attributes store each patient's state directly on the shared graph —
no parallel dict-of-dicts. Only symbols present in the PKN are lifted.

In [ ]:
graph_vertices = set(map(str, G.vertices()))
matched_patients = sorted(set(rna_contrast.columns) & set(tf_es.index) & set(kin_es.index))
assert matched_patients, 'No patients are shared across contrast and activity matrices.'

G.layers.set_aspects(['patient'], {'patient': matched_patients})
G.history.snapshot('after_patient_layers_defined')

rna_vertices = [g for g in rna_contrast.index if g in graph_vertices]
tf_vertices = [tf for tf in tf_es.columns if tf in graph_vertices]
kin_vertices = [k for k in kin_es.columns if k in graph_vertices]

for patient in matched_patients:
    aa = (patient,)
    G.add_vertices(sorted(set(rna_vertices) | set(tf_vertices) | set(kin_vertices)), layer=aa)
    for g in rna_vertices:
        v = rna_contrast.at[g, patient]
        if pd.notna(v):
            G.layers.set_vertex_layer_attrs(g, aa, rna_contrast=float(v))
    for tf in tf_vertices:
        v = tf_es.at[patient, tf]
        if pd.notna(v):
            G.layers.set_vertex_layer_attrs(tf, aa, tf_activity=float(v))
    for kin in kin_vertices:
        v = kin_es.at[patient, kin]
        if pd.notna(v):
            G.layers.set_vertex_layer_attrs(kin, aa, kinase_activity=float(v))

patient_layer_summary = pd.DataFrame({
    'metric': ['matched_patients', 'rna_vertices_in_graph', 'tf_vertices_in_graph', 'kinase_vertices_in_graph'],
    'value': [len(matched_patients), len(rna_vertices), len(tf_vertices), len(kin_vertices)],
})
patient_layer_summary.to_csv(TABLES / 'patient_layer_summary.csv', index=False)
G.history.snapshot('after_omics_loaded_into_layers')
patient_layer_summary

## Cohort selection

CORNETO scales superlinearly with input/output cardinality, so we solve over a
demonstration cohort (top-10% mutation burden) and a sensitivity cohort
(top-20%). Both are recorded and reused downstream.

In [ ]:
mut_mat = pd.read_parquet(MUT_MAT)
mut_sum = mut_mat.reindex(columns=matched_patients, fill_value=0).sum(axis=0).sort_values(ascending=False)

cohort_definitions = {}
for label, quantile in TOP_MUT_QUANTILES.items():
    cohort_definitions[label] = mut_sum[mut_sum >= float(mut_sum.quantile(quantile))].index.tolist()
if INCLUDE_ALL_MATCHED_COHORT:
    cohort_definitions['all_matched'] = list(mut_sum.index)

cohort_summary = pd.DataFrame([
    {'cohort_label': label, 'n_patients': len(members),
     'mean_mutation_burden': float(mut_sum.reindex(members).mean()) if members else np.nan,
     'median_mutation_burden': float(mut_sum.reindex(members).median()) if members else np.nan,
     'selection_note': 'demonstration cohort enriched for high mutation burden'
     if label.startswith('top') else 'sensitivity cohort'}
    for label, members in cohort_definitions.items()
]).sort_values('n_patients')
cohort_summary.to_csv(TABLES / 'cohort_summary.csv', index=False)

save_cohorts(cohort_definitions)
G.write(str(SNAPSHOT), overwrite=True)
print(f'cohorts: {[(k, len(v)) for k, v in cohort_definitions.items()]}')
cohort_summary